# 第三轮：从头训练的记忆结构对照
本轮仍使用合成频带帧。分别从同一随机初始化出发训练：`plastic`（多时间尺度活动＋可塑性连接）与 `recurrent`（同样活动宽度，关闭可塑性连接）。两者用同样步数和数据随机种子，但动态状态容量不同；比较只能说明关闭连接的影响，不能证明在严格等容量条件下的优越性。

依次运行全部单元格：获取 GitHub 代码、挂载 Drive、快速检查、训练两个模型、查看对比、打包。推荐 GPU；每个模型默认训练 1800 步。每 100 步写一次日志至 Drive。若 Colab 中断，请回传已有日志和完成的变体。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
assert (repo / 'trial3.py').is_file()
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('代码提交:', commit)
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/Parrot')
root = drive_root / 'trial3_results'
root.mkdir(parents=True, exist_ok=True)
print('结果目录:', root)


快速检查只验证程序能跑通，不代表训练成功。


In [ ]:
!python trial3.py --variant plastic --smoke --batch 4 --out /content/Parrot/trial3_smoke_plastic
!python trial3.py --variant recurrent --smoke --batch 4 --out /content/Parrot/trial3_smoke_recurrent


先训练带可塑性连接的模型。训练结束后再运行下一格。


In [ ]:
!python trial3.py --variant plastic --steps 1800 --batch 24 --eval 200 --out {root / 'plastic'}


再从头训练关闭可塑性连接的模型。随机种子和数据生成顺序与上一格相同。


In [ ]:
!python trial3.py --variant recurrent --steps 1800 --batch 24 --eval 200 --out {root / 'recurrent'}


In [ ]:
import json
from IPython.display import Audio, display
for variant in ('plastic', 'recurrent'):
    path = root / variant / 'metrics.json'
    report = json.loads(path.read_text())
    report['git_commit'] = commit
    path.write_text(json.dumps(report, indent=2))
    print('\n', variant, '回传增益 0.15：')
    for group in ('overall', 'repeated', 'distinct', 'length_3', 'length_4', 'length_5', 'length_6'):
        if group in report['conditions']['0.15']:
            item = report['conditions']['0.15'][group]
            print(group, 'n=', item['n'], '序列准确率=', round(item['sequence_exact'], 3),
                  '严格整段帧准确率=', round(item['full_frame_exact'], 3),
                  '提前发声率=', round(item['early_onset_rate'], 3))
    display(Audio(filename=str(root / variant / 'model.wav')))


最后把两个变体的日志、指标、音频和权重打包到 Google Drive，并将压缩包回传。


In [ ]:
import shutil
archive = shutil.make_archive(str(drive_root / 'trial3_results'), 'zip', root_dir=str(drive_root), base_dir='trial3_results')
print('请回传:', archive)
